## Necessary Imports

In [11]:
import os
from pathlib import Path

from haystack import Pipeline
from haystack.components.converters import TextFileToDocument
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder
from haystack.components.joiners import DocumentJoiner
from haystack.components.preprocessors import DocumentSplitter
from haystack.components.writers import DocumentWriter

from marqo_haystack import MarqoDocumentStore
from marqo_haystack.retriever import MarqoRetriever

## Writing the Data into the Marqo Document Store

In [6]:
HERE = Path(os.getcwd())
file_paths = [HERE / "data" / Path(name) for name in os.listdir("data")]

In [7]:
document_store = MarqoDocumentStore(768, collection_name="example-haystack-document-store")

2025-12-03 07:31:31,763 logger:'marqo' WARNING Your Marqo Python client requires a minimum Marqo version of 2.23.1 to function properly, but your Marqo version is 2.16.1. Please upgrade your Marqo instance to avoid potential errors. If you have already changed your Marqo instance but still get this warning, please restart your Python interpreter.


In [8]:
indexing_pipeline = Pipeline()
indexing_pipeline.add_component("converter", TextFileToDocument())
indexing_pipeline.add_component("splitter", DocumentSplitter())
indexing_pipeline.add_component("embedder", SentenceTransformersDocumentEmbedder())
indexing_pipeline.add_component("writer", DocumentWriter(document_store))
indexing_pipeline.connect("converter", "splitter")
indexing_pipeline.connect("splitter", "embedder")
indexing_pipeline.connect("embedder", "writer")

🚅 Components
  - converter: TextFileToDocument
  - splitter: DocumentSplitter
  - embedder: SentenceTransformersDocumentEmbedder
  - writer: DocumentWriter
🛤️ Connections
  - converter.documents -> splitter.documents (list[Document])
  - splitter.documents -> embedder.documents (list[Document])
  - embedder.documents -> writer.documents (list[Document])

In [ ]:
indexing_pipeline.run({"converter": {"sources": file_paths}})

In [10]:
document_store.count_documents()

516


### Retrieving the Documents

In [13]:
retrieval_pipeline = Pipeline()
retrieval_pipeline.add_component("text_embedder", SentenceTransformersTextEmbedder())
# Retrieve by embedding
retrieval_pipeline.add_component("embedding_retriever", MarqoRetriever(document_store=document_store, top_k=3))
# Retrieve by text
retrieval_pipeline.add_component("text_retriever", MarqoRetriever(document_store=document_store, top_k=3))
retrieval_pipeline.add_component("document_joiner", DocumentJoiner())

retrieval_pipeline.connect("text_embedder.embedding", "embedding_retriever.query")
retrieval_pipeline.connect("embedding_retriever.documents", "document_joiner.documents")
retrieval_pipeline.connect("text_retriever.documents", "document_joiner.documents")

🚅 Components
  - text_embedder: SentenceTransformersTextEmbedder
  - embedding_retriever: MarqoRetriever
  - text_retriever: MarqoRetriever
  - document_joiner: DocumentJoiner
🛤️ Connections
  - text_embedder.embedding -> embedding_retriever.query (list[float])
  - embedding_retriever.documents -> document_joiner.documents (List[Document])
  - text_retriever.documents -> document_joiner.documents (List[Document])

In [14]:
question = "Is black and white text boring?"
data = {
    "text_embedder": {"text": question},
    "text_retriever": {"query": question},
}

In [15]:
result = retrieval_pipeline.run(data=data)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]


In [17]:
result["document_joiner"]["documents"]

[Document(id=97f4f8361390be1b828cb1c884a6bccb4a704ad8cab06ef1e19ceac83aad0877, content: '*usr_06.txt*	For Vim version 9.0.  Last change: 2021 Nov 07
 
 		     VIM USER MANUAL - by Bram Mool...', meta: {'page_number': 1, 'split_id': 0, 'split_idx_start': 0, 'file_path': 'usr_06.txt', 'source_id': '3ec4d1038700fca622b15a71deae0ed2558783f96fc0c996ca14ca942d940227'}, score: 0.6045383500902562, embedding: vector of size 768),
 Document(id=d34b010686d6be5ac77cbc0f7454d69c8d45dbcc71a74fe8f442068f2d3124b9, content: 'hex colors and you can define new names for hex
 colors in |v:colornames| like so: >
 
 	let v:color...', meta: {'page_number': 1, 'split_id': 5, 'split_idx_start': 6803, 'file_path': 'usr_06.txt', 'source_id': '3ec4d1038700fca622b15a71deae0ed2558783f96fc0c996ca14ca942d940227'}, score: 0.6003298604976534, embedding: vector of size 768),
 Document(id=9b7263f0915c90bb84ede1ba5037ea1b5d22353ef244825ab37b529c217ab4ff, content: 'terminal
 that supports colors, the colors you see are mad